Trial with possibility framework

In [ ]:
def gpd_loglik(params, x):
    """Log-likelihood of the GPD (sum of log-densities), -inf outside support."""
    xi, sigma = params
    if sigma <= 0:
        return -np.inf
    if np.any(1.0 + xi * x / sigma <= 0):     # GPD support condition
        return -np.inf
    return np.sum(stats.genpareto.logpdf(x, c=xi, scale=sigma))


def box_prior(xi, sigma, xi_bounds=(0.0, 3.0), sigma_bounds=(0.0, 2.0)):
    """Possibilistic prior: f = 1 on the box, 0 outside. No normalisation needed."""
    in_box = (xi_bounds[0] <= xi <= xi_bounds[1]) and (sigma_bounds[0] <= sigma <= sigma_bounds[1])
    return 1.0 if in_box else 0.0


def possibility_posterior(xi, sigma, data, max_loglik,
                           xi_bounds=(0.0, 3.0), sigma_bounds=(0.0, 2.0)):
    """f(xi, sigma | x) = L(xi,sigma|x) / max_box L(.|x), restricted to the prior box."""
    if box_prior(xi, sigma, xi_bounds, sigma_bounds) == 0.0:
        return 0.0
    ll = gpd_loglik((xi, sigma), data)
    return float(np.exp(ll - max_loglik)) if np.isfinite(ll) else 0.0